<a href="https://colab.research.google.com/github/Amankumar1456/Amankumar1456/blob/main/openrouter_first_call.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Your first LLM call with OpenRouter

Goal: make one simple call, look at what comes back, then test **what an LLM can and cannot do** on its own (no tools, no memory, no internet).

**Before you run:** in Colab, open the 🔑 *Secrets* panel (left sidebar), make sure a secret named `OPENROUTER_API_KEY` exists, and switch on *Notebook access* for it.

Model: `nvidia/nemotron-3-ultra-550b-a55b:free` (free tier, so it can be rate-limited; the helper below retries). It is a large reasoning model, so replies can take a bit longer. To try another model later, change the `MODEL` line in the setup cell.

## 1. Setup

In [ ]:
import os, json, time, datetime, requests

# Colab keeps the key in Secrets. Fall back to an environment variable if you run this elsewhere.
try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    API_KEY = os.environ.get("OPENROUTER_API_KEY")

assert API_KEY, "No key found. Add OPENROUTER_API_KEY in Colab Secrets and enable notebook access."

MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
URL   = "https://openrouter.ai/api/v1/chat/completions"

def call(messages, **params):
    """Send one request. `messages` is a list of {role, content}. Returns the full JSON response."""
    body = {"model": MODEL, "messages": messages, **params}
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    for attempt in range(4):                      # free models sometimes return 429 (busy)
        r = requests.post(URL, headers=headers, json=body, timeout=120)
        if r.status_code == 429:
            time.sleep(2 ** attempt * 2)
            continue
        break
    data = r.json()
    if r.status_code != 200 or "error" in data:
        raise RuntimeError(f"HTTP {r.status_code}: {data}")
    return data

def ask(prompt, **params):
    """Shortcut: one user message in, the reply text out."""
    resp = call([{"role": "user", "content": prompt}], **params)
    return resp["choices"][0]["message"].get("content") or ""

print("Ready. Using model:", MODEL)

Ready. Using model: nvidia/nemotron-3-ultra-550b-a55b:free


## 2. The simplest possible call

An LLM call is: **a list of messages in, one message out.** That's the whole interface.

In [ ]:
resp = call([{"role": "user", "content": "In one sentence, what is an API?"}])
print(resp["choices"][0]["message"]["content"])

An API (Application Programming Interface) is a set of rules and protocols that allows different software applications to communicate and exchange data with each other.


### What actually came back?
The text is only one field. The rest tells you which model answered and what it cost in tokens.

In [ ]:
msg = resp["choices"][0]
print("model        :", resp.get("model"))
print("finish_reason:", msg.get("finish_reason"), " (stop = it finished; length = it ran out of max_tokens)")
print("usage        :", resp.get("usage"))
print()
print(json.dumps(resp, indent=2)[:1500])

model        : nvidia/nemotron-3-ultra-550b-a55b:free
finish_reason: stop  (stop = it finished; length = it ran out of max_tokens)
usage        : {'prompt_tokens': 25, 'completion_tokens': 43, 'total_tokens': 68, 'cost': 0, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}, 'completion_tokens_details': {'reasoning_tokens': 13, 'image_tokens': 0, 'audio_tokens': 0}}

{
  "id": "gen-1794826215-MbMKM91EPB6gUyEyZtte",
  "object": "chat.completion",
  "created": 1794826215,
  "model": "nvidia/nemotron-3-ultra-550b-a55b:free",
  "provider": "Nvidia",
  "system_fingerprint": null,
  "service_tier": null,
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "native_finish_reason": "stop",
      "message": {
        "role": "assistant",
        "co

---
# ✅ What it CAN do

Pure text in, text out. Anything that is "work with language" is fair game.

### Rewrite and change tone

In [ ]:
print(ask("Rewrite this so it is polite but still firm: 'Send me the report now.'"))

Here are a few options, depending on the context and your relationship with the recipient:

**Standard Professional (Best for most situations)**
> "Please send me the report as soon as possible."

**Slightly More Formal**
> "Kindly provide the report at your earliest convenience."

**If There Is a Specific Deadline (Firmness comes from clarity)**
> "Please send the report by [specific time/date] so we can stay on schedule."

**If You Need It Immediately (Urgent but polite)**
> "This is time-sensitive—please send the report right away."

**If You Are Following Up (Implies they should have already sent it)**
> "Just following up on the report—please send it over today."


### Summarize

In [ ]:
text = """
The quarterly review meeting ran 40 minutes over. Priya raised concerns that the onboarding flow
loses about a third of users at the payment step. Marcus said a fix would need two sprints because
the payment provider's SDK is outdated. The team agreed to ship a stopgap (a saved-cart reminder
email) this month and schedule the SDK upgrade for Q1. Action: Priya drafts the email by Friday.
"""
print(ask(f"Summarize in two bullet points:\n{text}"))

- **Onboarding drop-off identified:** The review ran 40 min over; Priya flagged that ~⅓ of users abandon at the payment step, and Marcus estimated a full fix needs two sprints because the payment provider’s SDK is outdated.  
- **Immediate stopgap & roadmap:** Team will ship a saved‑cart reminder email this month (Priya to draft by Friday) and schedule the SDK upgrade for Q1.


### Pull structured data out of messy text
This is one of the most useful tricks: turn free text into JSON your code can use.

In [ ]:
receipt = "Paid $18.50 to Blue Bottle Coffee on 3 March for 2 lattes and a croissant. Card ending 4242."
out = ask(
    "Extract vendor, amount (number), currency, and date from this text. "
    "Reply with ONLY a JSON object, no explanation.\n\n" + receipt
)
print("raw reply:", out)

# Models sometimes wrap JSON in ```json fences, so strip them before parsing.
clean = out.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
data = json.loads(clean)
print("parsed   :", data, "->", type(data["amount"]))

raw reply: {
  "vendor": "Blue Bottle Coffee",
  "amount": 18.5,
  "currency": "USD",
  "date": "3 March"
}
parsed   : {'vendor': 'Blue Bottle Coffee', 'amount': 18.5, 'currency': 'USD', 'date': '3 March'} -> <class 'float'>


### Write code
Note: it **writes** the code. It does not **run** it. You would have to.

In [ ]:
print(ask("Write a short Python function is_palindrome(s) that ignores case and punctuation. Code only."))

```python
import re

def is_palindrome(s):
    cleaned = re.sub(r'[^a-z0-9]', '', s.lower())
    return cleaned == cleaned[::-1]
```


### Reason step by step
Some models can think before answering. Turn it on with the `reasoning` parameter. The thinking comes back separately from the answer.

In [ ]:
puzzle = "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much is the ball?"
resp = call([{"role": "user", "content": puzzle}], reasoning={"enabled": True})
m = resp["choices"][0]["message"]
print("ANSWER:", m.get("content"))
print()
print("Has hidden reasoning?", bool(m.get("reasoning_details") or m.get("reasoning")))
print("Reasoning tokens used:", (resp.get("usage", {}).get("completion_tokens_details") or {}).get("reasoning_tokens"))

ANSWER: The ball costs **$0.05** (5 cents).

Here is the breakdown:
*   **Ball:** $0.05
*   **Bat:** $1.05 (which is $1.00 more than the ball)
*   **Total:** $1.10

*The intuitive answer is often 10 cents, but that would make the bat $1.10 and the total $1.20.*

Has hidden reasoning? True
Reasoning tokens used: 171


---
# ❌ What it CANNOT do (on its own)

Each test below sets a trap. Read the output and compare it with the truth.

### 1. It has no memory
Each call is separate. The model only knows what you send in *that* request.

In [ ]:
print("Call 1:", ask("My name is Rutvik and I live in Bangalore. Remember that."))
print()
print("Call 2:", ask("What is my name and where do I live?"))

Call 1: Got it, Rutvik! I'll remember that you live in Bangalore. How can I help you today?

Call 2: I don't know. As an AI, I don't have access to your personal information, identity, or location unless you share it with me during our conversation.


The 'memory' in chat apps is just the app **re-sending the whole conversation** every time. You can do the same:

In [ ]:
history = [
    {"role": "user", "content": "My name is Rutvik and I live in Pune. Remember that."},
    {"role": "assistant", "content": "Got it, Rutvik from Pune."},
    {"role": "user", "content": "What is my name and where do I live?"},
]
print(call(history)["choices"][0]["message"]["content"])

Your name is **Rutvik** and you live in **Pune**.


### 2. It does not know today's date or the latest news
Its knowledge stops at training time and it has no internet.

In [ ]:
print("Model says :", ask("What is today's date, and what was the top news headline this morning?"))
print()
print("Truth      :", datetime.date.today().isoformat())

Model says : I do not have access to real-time information, including the current date or live news feeds.

**To find today's date:** Please check the clock on your device (computer, phone, tablet).

**To find the top news headlines:** I recommend visiting a reputable news website directly, such as:
*   **BBC News** / **Reuters** / **Associated Press** (for international wire coverage)
*   **The New York Times** / **The Washington Post** / **The Wall Street Journal** (US focus)
*   **The Guardian** (UK/International focus)
*   **Al Jazeera** (Middle East/International focus)
*   **Google News** or **Apple News** (for aggregated headlines tailored to your region)

Truth      : 2026-11-16


### 3. It cannot open your files or take actions
We create a file. The model never sees it unless *you* paste the contents in.

In [ ]:
with open("sample.txt", "w") as f:
    f.write("The secret word is MANGO-42.")

print("Model says :", ask("Read the file sample.txt in my current folder and tell me the secret word."))
print()
print("Truth      :", open("sample.txt").read())

Model says : I cannot access files on your local computer or server. I don't have the ability to read your file system.

To find the secret word, you'll need to open the file yourself using a text editor (like Notepad, VS Code, TextEdit) or a terminal command:

**Windows (Command Prompt):**
```cmd
type sample.txt
```

**Windows (PowerShell):**
```powershell
Get-Content sample.txt
```

**Mac / Linux (Terminal):**
```bash
cat sample.txt
```

Once you open it, you'll see the secret word inside.

Truth      : The secret word is MANGO-42.


### 4. Exact counting and big arithmetic are shaky
It predicts likely text, it does not calculate. Always verify with code.

In [ ]:
sentence = "The quick brown fox jumps over the lazy dog while seven eleven elephants entertain everyone."
a, b = 48271, 90817

model_count = ask(f"How many times does the letter 'e' appear in: {sentence}  Reply with just a number.")
model_prod  = ask(f"What is {a} x {b}? Reply with just the number, no commas.")

true_count, true_prod = sentence.lower().count("e"), a * b
print(f"letter 'e'  -> model: {model_count.strip():>12} | python: {true_count}")
print(f"{a} x {b} -> model: {model_prod.strip():>12} | python: {true_prod}")
print("\nIf both matched this time, run the cell again or pick bigger numbers. The point: check, do not trust.")

letter 'e'  -> model:           16 | python: 16
48271 x 90817 -> model:   4383827407 | python: 4383827407

If both matched this time, run the cell again or pick bigger numbers. The point: check, do not trust.


### 5. It can confidently make things up
We ask about a pandas method that **does not exist**.

In [ ]:
import pandas as pd
print(ask("Explain what pandas.DataFrame.auto_clean(strategy='smart') does and show an example."))
print()
print("Does it exist? ->", hasattr(pd.DataFrame, "auto_clean"))

**`pandas.DataFrame.auto_clean()` does not exist in the standard pandas library.**

There is no method named `auto_clean` attached to `pandas.DataFrame` in any official release (as of pandas 2.2+). It is likely you encountered this in one of the following contexts:

1.  **A third-party extension library** (e.g., `pyjanitor`, `dataprep`, `autogluon`, or a custom internal company package).
2.  **A helper function defined in a tutorial/blog post** (often named `auto_clean` for demonstration purposes).
3.  **A hallucinated/assumed API** by an LLM or autocomplete tool.

---

### The Closest Real Alternative: `pyjanitor`
The most popular library adding "cleaning" methods directly to the DataFrame API is **`pyjanitor`**. It registers methods via `pd.DataFrame.register_method` (or simply `import janitor`).

It has a method called **`clean_names()`** (often what people want) but **no `auto_clean()`**.

#### Example using `pyjanitor` (Real Code)
```python
import pandas as pd
import janitor  # re

### 6. The same question can give different answers
By default there is randomness (`temperature`). Lower it and fix a `seed` for more repeatable output.

In [ ]:
q = "Suggest one name for a coffee shop. Reply with just the name."
print("Default (temperature=1):", [ask(q).strip() for _ in range(3)])
print("temperature=0, seed=7  :", [ask(q, temperature=0, seed=7).strip() for _ in range(3)])

Default (temperature=1): ['The Daily Grind', 'The Daily Grind', 'The Daily Grind']
temperature=0, seed=7  : ['The Daily Roast', 'The Daily Grind', 'The Daily Grind']


---
# 🔧 Tasks that cannot run without tools

Some questions need **live data from the outside world**: weather, prices, headlines, new research. The model has none of it, so it will either refuse or, worse, invent something plausible.

For each task below we do three things:
1. **Model alone:** ask the question plainly and watch what happens.
2. **Tool result:** fetch the real answer with ordinary code from a free public API (no keys needed). This code is the "tool".
3. **Model + data:** paste the tool result into the prompt and ask again.

Notice who does what: **code fetches, the model explains.**

In [ ]:
import xml.etree.ElementTree as ET, csv, io

HDR = {"User-Agent": "Mozilla/5.0 (colab-demo)"}

def get(url, **kw):
    """A tiny 'tool': fetch a URL from the internet."""
    r = requests.get(url, headers=HDR, timeout=30, **kw)
    r.raise_for_status()
    return r

def ask_with_data(question, data):
    """Manual 'tool use': we paste the fetched data into the prompt and tell the model to rely on it."""
    return ask(f"Answer using ONLY the data below.\n\nDATA:\n{data}\n\nQUESTION: {question}")

### Tool task 1: Current weather

In [ ]:
q = "What is the temperature in Pune right now, and is it windy?"
print("① Model alone :", ask(q))

weather = get("https://api.open-meteo.com/v1/forecast",
              params={"latitude": 18.52, "longitude": 73.86,
                      "current": "temperature_2m,wind_speed_10m,relative_humidity_2m"}).json()["current"]
print("\n② Tool result :", weather)
print("\n③ Model + data:", ask_with_data(q, weather))

① Model alone : I don't have access to real-time data, so I cannot tell you the current temperature or wind conditions in Pune.

To get the latest weather, please check a reliable weather source such as:

*   **Google Weather** (search "weather Pune")
*   **India Meteorological Department (IMD)** website or app
*   **Weather.com** or **AccuWeather**
*   **Smartphone weather apps** (Apple Weather, Google Weather, etc.)


ReadTimeout: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=30)

### Tool task 2: Stock price

In [ ]:
q = "What is Apple's (AAPL) latest stock price, and how much did it move in the last session?"
print("① Model alone :", ask(q))

try:   # Yahoo Finance's public chart endpoint
    res = get("https://query1.finance.yahoo.com/v8/finance/chart/AAPL", params={"interval": "1d", "range": "5d"}).json()["chart"]["result"][0]
    closes = [c for c in res["indicators"]["quote"][0]["close"] if c]
    quote = {"symbol": "AAPL", "latest_price": round(res["meta"]["regularMarketPrice"], 2),
             "previous_close": round(closes[-2], 2), "currency": res["meta"]["currency"],
             "as_of": datetime.datetime.fromtimestamp(res["meta"]["regularMarketTime"]).isoformat()}
except Exception as e:   # fallback source if Yahoo blocks this machine
    row = next(csv.DictReader(io.StringIO(get("https://stooq.com/q/l/?s=aapl.us&f=sd2t2ohlcv&h&e=csv").text)))
    quote = {"symbol": row["Symbol"], "date": row["Date"], "open": row["Open"], "close": row["Close"]}

print("\n② Tool result :", quote)
print("\n③ Model + data:", ask_with_data(q, quote))

① Model alone : I do not have access to real-time financial data, so I cannot provide Apple's (AAPL) current stock price or its movement from the last trading session.

For the most up-to-date information, please check a reliable financial source such as:

*   **Google Finance** (search "AAPL")
*   **Yahoo Finance**
*   **Bloomberg** or **Reuters**
*   **Your brokerage platform** (Fidelity, Schwab, Robinhood, etc.)
*   **NASDAQ.com** or **NYSE.com**

② Tool result : {'symbol': 'AAPL', 'latest_price': 356.27, 'previous_close': 354.88, 'currency': 'USD', 'as_of': '2026-11-13T21:00:01'}

③ Model + data: Apple's (AAPL) latest stock price is $356.27 (USD) as of 2026-11-13T21:00:01. In the last session, the stock moved up by $1.39 (from a previous close of $354.88).


### Tool task 3: Today's news

In [ ]:
q = "What are the top 3 news headlines right now?"
print("① Model alone :", ask(q))

root = ET.fromstring(get("https://feeds.bbci.co.uk/news/rss.xml").content)
headlines = [{"title": i.findtext("title"), "published": i.findtext("pubDate")} for i in root.iter("item")][:5]
print("\n② Tool result :", json.dumps(headlines, indent=1))
print("\n③ Model + data:", ask_with_data(q, headlines))

① Model alone : I don't have real-time access to the internet or live news feeds, so I cannot provide current headlines. My knowledge is current only up to my training cutoff.

For the latest top headlines, I recommend checking a trusted news source directly, such as:
*   **BBC News** / **Reuters** / **Associated Press** (for global wire coverage)
*   **The New York Times** / **The Washington Post** / **The Wall Street Journal** (for US focus)
*   **Google News** / **Apple News** (for personalized aggregates)
*   Your preferred local news outlet's website or app.

② Tool result : [
 {
  "title": "Create more jobs for under-25s by cutting National Insurance, key report urges",
  "published": "Mon, 16 Nov 2026 10:51:45 GMT"
 },
 {
  "title": "UK lost more than it gained from Brexit, Burnham to say ahead of EU summit",
  "published": "Mon, 16 Nov 2026 11:27:53 GMT"
 },
 {
  "title": "'I'd throw up in the car park': How school anxiety is harming children's futures",
  "published": "Mon, 16

### Tool task 4: Research (latest papers)
The model may list papers that sound real. Check them against arXiv.

In [ ]:
q = "List the 3 most recent arXiv papers about 'retrieval augmented generation' with titles and submission dates."
print("① Model alone :", ask(q))

feed = ET.fromstring(get("https://export.arxiv.org/api/query",
                         params={"search_query": 'all:"retrieval augmented generation"',
                                 "sortBy": "submittedDate", "sortOrder": "descending", "max_results": 3}).content)
ns = {"a": "http://www.w3.org/2005/Atom"}
papers = [{"title": " ".join(e.findtext("a:title", namespaces=ns).split()),
           "submitted": e.findtext("a:published", namespaces=ns)[:10]} for e in feed.findall("a:entry", ns)]
print("\n② Tool result :", json.dumps(papers, indent=1))
print("\n③ Model + data:", ask_with_data(q, papers))

① Model alone : As an AI, I **do not have real-time access to the live arXiv database** (my knowledge cutoff is earlier than "right now"). I cannot browse the internet to fetch papers submitted in the last few hours or days.

To get the **absolute three most recent papers** right this second, you must check arXiv directly using one of these methods:

### 1. Direct arXiv Search Link (Best Method)
Click this pre-filtered link sorted by **submission date (newest first)**:
🔗 **[arXiv Search: "retrieval augmented generation" sorted by date](https://arxiv.org/search/?query=%22retrieval+augmented+generation%22&searchtype=all&abstracts=show&order=-announced_date_first&size=50)**

### 2. How to search manually
1. Go to **[arxiv.org](https://arxiv.org)**.
2. Type `"retrieval augmented generation"` (with quotes for exact phrase) in the search bar.
3. In the dropdown next to the search button, select **"Submission date"** (or "Announced date first" -> Descending).
4. The top 3 results are your ans

### What just happened

| | Model alone | With a tool |
|---|---|---|
| Weather, price, headlines, papers | Refuses, guesses, or invents | Correct, because the facts came from outside |
| Who found the facts? | Nobody | **Our code**, not the model |

Here **we** decided which API to call and pasted the result in by hand. An **agent** automates exactly that: the model *chooses* which tool to call, the program runs it, the result goes back into the conversation, and the model decides what to do next (research is usually several rounds of search, read, search again). That loop is the next step.

---
# Summary

| It can | It cannot (alone) | What fixes it |
|---|---|---|
| Write, rewrite, summarize, translate | Remember past calls | You re-send history, or add a memory store |
| Extract structure from text | Know today's date, live weather, prices, news or new research | **Tools** (APIs, search) or paste the info into the prompt |
| Write and explain code | Open files or take actions | **Tools** (file read, run code, APIs) |
| Reason through problems | Count or calculate reliably | Let it call a calculator / run Python |
| Follow instructions on what you paste in | Admit what it doesn't know | Give it sources to cite; verify outputs |

Everything in the right column is what **agents** add on top: a loop that lets the model call tools, see the results, and keep going. That is the next step.